# 03 - Lakebase: refresh pgvector index + verify operational serving
Final task of the `smb-cashflow-copilot` job (after the UC -> Lakebase synced tables refresh).
Rebuilds `app.note_vectors` (native `vector(1024)` + HNSW) from the synced embeddings, then
checks the serving tables the app reads and times a point lookup + a semantic search.

In [ ]:
%pip install -q psycopg2-binary
%restart_python

In [ ]:
import time, psycopg2
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
ENDPOINT = "projects/smb-cashflow/branches/production/endpoints/primary"
ep = w.api_client.do("GET", f"/api/2.0/postgres/{ENDPOINT}")
host = ep["status"]["hosts"]["host"]
token = w.api_client.do("POST", "/api/2.0/postgres/credentials", body={"endpoint": ENDPOINT})["token"]
user = w.current_user.me().user_name
conn = psycopg2.connect(host=host, dbname="cashflow", user=user, password=token, sslmode="require")
conn.autocommit = True
cur = conn.cursor()
print("connected to Lakebase", host.split(".")[0], "as", user)

connected to Lakebase <lakebase-endpoint> as <user>


In [ ]:
t = time.time()
cur.execute("REFRESH MATERIALIZED VIEW CONCURRENTLY app.note_vectors")
print(f"refreshed app.note_vectors in {time.time() - t:.1f}s")
for tbl in ["smb_cashflow.lb_account_cashflow_health", "smb_cashflow.lb_invoices", "smb_cashflow.lb_bills",
            "smb_cashflow.lb_success_agents", "smb_cashflow.lb_agent_portfolio", "smb_cashflow.lb_copilot_briefings",
            "smb_cashflow.lb_account_notes_embedded", "app.note_vectors", "app.agent_actions", "app.copilot_audit"]:
    cur.execute(f"SELECT count(*) FROM {tbl}")
    print(f"{tbl:<42} {cur.fetchone()[0]:>7} rows")

refreshed app.note_vectors in 1.0s
smb_cashflow.lb_account_cashflow_health       1500 rows
smb_cashflow.lb_invoices                     29509 rows
smb_cashflow.lb_bills                        17198 rows
smb_cashflow.lb_success_agents                  12 rows
smb_cashflow.lb_agent_portfolio                 12 rows
smb_cashflow.lb_copilot_briefings              304 rows
smb_cashflow.lb_account_notes_embedded        3063 rows
app.note_vectors                              3063 rows
app.agent_actions                                2 rows
app.copilot_audit                                6 rows


In [ ]:
t = time.time()
cur.execute("""SELECT account_id, business_name, current_balance, overdue_ar, runway_days, shortfall_prob, risk_tier
               FROM smb_cashflow.lb_account_cashflow_health WHERE account_id = 'SMB-0112'""")
row = cur.fetchone()
print(f"point lookup {1000 * (time.time() - t):.1f} ms ->", row)

cur.execute("SELECT emb FROM app.note_vectors WHERE account_id = 'SMB-0112' LIMIT 1")
q = cur.fetchone()[0]
t = time.time()
cur.execute("""SELECT account_id, left(note_text, 70), round((emb <=> %s::vector)::numeric, 3)
               FROM app.note_vectors WHERE account_id <> 'SMB-0112' ORDER BY emb <=> %s::vector LIMIT 3""", (q, q))
res = cur.fetchall()
print(f"pgvector HNSW top-3 search {1000 * (time.time() - t):.1f} ms")
for r in res:
    print("  ", r)
conn.close()

point lookup 5.3 ms -> ('SMB-0112', 'Granite Goods', 21500.0, 89040.0, 39.0, 0.7447, 'High')
pgvector HNSW top-3 search 9.0 ms
   ('SMB-0736', 'Owner flagged a cash crunch; payroll due next week and receivables are', Decimal('0.267'))
   ('SMB-0718', 'Owner flagged a cash crunch; payroll due next week and receivables are', Decimal('0.267'))
   ('SMB-0656', 'Owner flagged a cash crunch; payroll due next week and receivables are', Decimal('0.267'))
